# 🔥 Module 9: Forest Segmentation Using U-Net

Welcome to **Module 9** of the **Deforestation Detection from Satellite Images** project!

### 🎯 Learning Objectives
1. **Semantic Segmentation**: Understand why pixel-level dense classification is essential for deforestation mapping.
2. **Encoder-Decoder Architecture**: Implement the classic **U-Net** in PyTorch adapted for 5-band satellite inputs.
3. **Skip Connections**: Understand how high-resolution encoder features bypass downsampling to reconstruct sharp canopy boundaries.
4. **Compound Loss (BCE + Dice)**: Optimize models using a hybrid objective that handles class imbalance.
5. **Evaluate IoU & Dice Score**: Quantify segmentation overlap on unseen test scenes.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch

# Add project root to sys.path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from modules.module_09_unet_segmentation.unet_model import ForestUNet
from modules.module_09_unet_segmentation.losses_metrics import CombinedBCEDiceLoss, compute_iou_dice
from modules.module_09_unet_segmentation.segmentation_dataset import create_segmentation_dataloaders
from modules.module_09_unet_segmentation.trainer import train_unet, evaluate_unet

%matplotlib inline
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Module 9 U-Net environment ready on device: [{device}]")

## 1. Load Dense Segmentation DataLoaders

We load paired satellite imagery and ground-truth forest masks (1 = Forest, 0 = Non-Forest).

In [ ]:
train_loader, val_loader, test_loader = create_segmentation_dataloaders(
    dataset_root="../dataset",
    tile_size=128,
    batch_size=8,
    max_train_tiles=24,
    max_val_tiles=8
)
print(f"Train tiles: {len(train_loader.dataset)} | Val tiles: {len(val_loader.dataset)} | Test tiles: {len(test_loader.dataset)}")

## 2. Visualize Input Satellite Tile & Ground-Truth Forest Mask

In [ ]:
def make_rgb(tensor_5ch):
    p = tensor_5ch.cpu().numpy()
    def s(arr):
        p2, p98 = np.percentile(arr, (2, 98))
        return np.clip((arr - p2) / (p98 - p2 + 1e-8), 0.0, 1.0)
    return np.stack([s(p[2]), s(p[1]), s(p[0])], axis=-1)

batch_x, batch_y = next(iter(train_loader))
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)
axes[0].imshow(make_rgb(batch_x[0]))
axes[0].set_title("Satellite Tile (RGB)", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(batch_y[0, 0].numpy(), cmap="Greens_r")
axes[1].set_title("Ground-Truth Forest Mask", fontweight="bold")
axes[1].axis("off")
plt.show()

## 3. Instantiate & Train ForestUNet

The U-Net features:
- 3 Encoder contracting stages capturing semantic context
- Bottleneck (128 channels)
- 3 Decoder expansive stages with skip connections

In [ ]:
model = ForestUNet(in_channels=5, out_channels=1, base_features=16)
trained_model, history = train_unet(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=5,
    lr=1e-3,
    device=device
)

## 4. Plot Convergence (Loss, IoU, Dice Score)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
epochs_r = range(1, len(history["train_loss"]) + 1)

ax1.plot(epochs_r, history["train_loss"], 'b-o', label='Train Loss')
ax1.plot(epochs_r, history["val_loss"], 'r--s', label='Val Loss')
ax1.set_title('Combined BCE + Dice Loss', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend()

ax2.plot(epochs_r, history["val_iou"], 'g-o', label='Val IoU (%)')
ax2.plot(epochs_r, history["val_dice"], 'm--s', label='Val Dice (%)')
ax2.set_title('Validation IoU and Dice Overlap', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend()
plt.show()

## 5. Evaluate on Unseen Test Imagery

We benchmark pixel-level metrics: IoU, Dice score, Precision, and Recall.

In [ ]:
metrics = evaluate_unet(trained_model, test_loader, device=device)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k:<20}: {v*100:>6.2f}%")
    else:
        print(f"{k:<20}: {v}")

## 6. Dense Forest Segmentation Prediction

Inspect the continuous sigmoid probability map and the final binary forest mask.

In [ ]:
test_x, test_y = next(iter(test_loader))
sample_x = test_x[0:1].to(device)
sample_gt = test_y[0, 0].numpy()

trained_model.eval()
with torch.no_grad():
    logits = trained_model(sample_x)
    probs = torch.sigmoid(logits)[0, 0].cpu().numpy()
    pred_bin = (probs >= 0.5).astype(np.float32)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), constrained_layout=True)
axes[0].imshow(make_rgb(sample_x[0]))
axes[0].set_title("1. Satellite Tile (RGB)", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(sample_gt, cmap="Greens_r")
axes[1].set_title("2. Ground Truth Mask", fontweight="bold")
axes[1].axis("off")

im_p = axes[2].imshow(probs, cmap="viridis", vmin=0, vmax=1)
axes[2].set_title("3. U-Net Sigmoid Probabilities", fontweight="bold")
axes[2].axis("off")
fig.colorbar(im_p, ax=axes[2], fraction=0.046, pad=0.04)

axes[3].imshow(pred_bin, cmap="Greens_r")
axes[3].set_title(f"4. Predicted Forest Mask\n(IoU: {metrics['iou']*100:.1f}%)", fontweight="bold", color="darkgreen")
axes[3].axis("off")
plt.show()